# 00 — Pipeline smoke test

End-to-end demo on a short, cheap window around a known strong-flare / geomagnetic-storm day
(**2024-05-11**, padded by ±1 day): fetch BTS flights + OMNI + GOES X-ray → align → one of each plot type.

* **Live mode** (default) uses the real free sources — no keys needed. Note that BTS ships whole
  *months* (one ~25 MB zip → ~250 MB CSV, cached under `data/raw/bts/`), so the first run takes a few minutes.
* **Offline mode** (`OFFLINE_DEMO=1`) skips all network access and builds *synthetic* frames with the same
  schemas, to check the plumbing. Plots made in that mode are **not real data**.

No statistics are computed here — this only demonstrates the data infrastructure.

In [ ]:
import os, tempfile
from pathlib import Path

OFFLINE_DEMO = os.environ.get("OFFLINE_DEMO", "0") == "1"
START, END = "2024-05-10", "2024-05-12"
AIRPORTS = "ATL,ORD,DEN,LAX,JFK,ANC"   # small subset to keep the demo light; use "top20" for the study

if OFFLINE_DEMO:
    os.environ["SWE_DATA_DIR"] = tempfile.mkdtemp(prefix="swe_demo_")  # keep ./data untouched
print("offline demo:", OFFLINE_DEMO)

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from swe_airline_delay import storage
from swe_airline_delay.align import build_panel
from swe_airline_delay.config import resolve_airports
from swe_airline_delay.flight_delay import bts
from swe_airline_delay.plotting import figures, style
from swe_airline_delay.plotting.sample_data import make_synthetic
from swe_airline_delay.space_weather import goes_xray, omni

style.use_style()
airports = resolve_airports(AIRPORTS)
airports[["iata", "name", "lat", "lon"]]

## 1. Fetch (idempotent — re-running skips anything already in the manifest)

In [ ]:
if not OFFLINE_DEMO:
    bts.fetch(START, END, airports=AIRPORTS)            # BTS On-Time Performance (month chunk)
    omni.fetch(START, END, high_res=False)               # NASA OMNIWeb hourly via CDAWeb
    goes_xray.fetch(START, END)                          # NOAA NCEI archive / SWPC recent
    print(storage.Manifest().summary().to_string(index=False))
else:
    # Synthetic stand-ins with the same schemas (NOT real observations)
    d, idx, _ = make_synthetic(days=3, airports=list(airports["iata"]), start=START, seed=1)
    storage.write_parquet(d, "interim", "bts", 2024, "ALL", "2024-05")
    long = idx[["kp", "flow_speed"]].reset_index().melt("time", var_name="variable", value_name="value")
    long["resolution"] = "1h"
    storage.write_parquet(long.set_index("time"), "interim", "omni", 2024, "global", "2024-05_1h")
    omni.consolidate()
    t = pd.date_range(START, periods=72 * 60, freq="1min", tz="UTC", name="time")
    f = np.full(len(t), 2e-7)
    peak = 36 * 60 + 83
    k = np.arange(len(t)) - peak
    f = np.maximum(f, 2e-7 + 5.2e-5 * np.where(k < 0, np.clip(1 + k / 15, 0, 1), np.exp(-np.clip(k, 0, None) / 15)))
    storage.write_parquet(pd.DataFrame({"xrsa_flux": f / 10, "xrsb_flux": f}, index=t), "interim", "goes_xrs", 2024, "global", "2024-05-11")

## 2. Load tidy tables

In [ ]:
flights = bts.load(START, END, airports=AIRPORTS) if not OFFLINE_DEMO else storage.read_parquet("interim", "bts")
long = omni.load(START, END)
omni_h = omni.to_wide(long, "1h")
xray = goes_xray.load(START, END)
print(f"flights: {len(flights):,} rows | OMNI hourly: {omni_h.shape} | GOES XRS 1-min: {xray.shape}")
flights.head()

## 3. Flares from the X-ray flux (M-class and above) → event times

In [ ]:
flares = goes_xray.extract_flares(xray, min_class="M")
flares

## 4. Align (merge only)

In [ ]:
panel = build_panel(flights, {"omni1h": omni_h, "xrs": xray[["xrsa_flux", "xrsb_flux"]]}, tolerance="1h")
panel[["airport", "scheduled_dep", "dep_delay_min", "cancelled", "omni1h_kp", "xrs_xrsb_flux"]].head()

## 5. One of each figure type

In [ ]:
figures.plot_timeseries(flights, omni_h, "kp")
plt.show()

In [ ]:
figures.plot_dose_response(panel, "omni1h_kp", bins=5)
plt.show()

In [ ]:
events = flares["peak_time"] if len(flares) else pd.DatetimeIndex([pd.Timestamp("2024-05-11 01:23", tz="UTC")])
figures.plot_superposed_epoch(events, flights, window=(-3, 6))
plt.show()

In [ ]:
burden = flights.groupby("airport")["dep_delay_min"].mean().rename("mean_dep_delay_min")
amap = airports.merge(burden, left_on="iata", right_index=True)
figures.plot_airport_map(amap, color_by="mean_dep_delay_min", draw_features=not OFFLINE_DEMO)
plt.show()